# Benchmark ASR baoulé (bci) — Omnilingual vs Simba-S

Protocole : `docs/benchmarks/0003-asr-baoule-evaluation.md` — Wourri, issue #473.

**But** : mesurer le **CER/WER réel** de la transcription baoulé sur du vrai baoulé
(Common Voice `bci`, split *test*), pour choisir la base de fine-tuning.

**Runtime** : *Exécution → Modifier le type d'exécution → **T4 GPU***.
**Aucun token Hugging Face requis** (dataset public CC0).

Modèles : Omnilingual CTC **300M** & **1B** (ciblage `bci_Latn`) vs **Simba-S**
(contrôle — voir le garde-fou « langue de sortie »).

> ⚠️ Exécuter les cellules **dans l'ordre**. Omnilingual impose un **redémarrage de
> session** après son install (cf. benchmark 0002).

In [ ]:
# Cellule 1 — verifier le GPU + la version de Python
# ⚠️ Omnilingual (fairseq2n 0.6) n'existe QU'EN Python 3.10 / 3.11 / 3.12 (pas 3.13).
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout
print(out or "PAS DE GPU -> Colab: Execution > T4 GPU  |  Kaggle: Settings > Accelerator > GPU T4 x2")
print("Python:", sys.version.split()[0])
if sys.version_info[:2] > (3, 12):
    print("\n>>> STOP: Python", sys.version.split()[0], "> 3.12 -> le stack Omnilingual NE s'installera PAS.")
    print(">>> Kaggle est en general en 3.11 (OK). Colab actuel est en 3.13 (KO).")
else:
    print(">>> Python compatible Omnilingual. Continue.")

## Étape 1 — Omnilingual (M1 / M2)

Installe le stack Omnilingual (versions figées du benchmark 0002, ~5–7 min), **puis
REDÉMARRE LA SESSION** : *Exécution → Redémarrer la session* (surtout **PAS**
« Réinitialiser tout l'environnement »). Reprends ensuite à la **cellule 3**.

In [ ]:
# Cellule 2 — install Omnilingual (versions figees, benchmark 0002) + libs d'eval
# fairseq2 EPINGLE : omnilingual-asr (0.1.0/0.2.0) exige fairseq2[arrow]>=0.5.2,<=0.6.
# Un `fairseq2` nu prend desormais la derniere (0.8.x) -> ResolutionImpossible.
# On epingle la combo VALIDEE (benchmark 0002) : omnilingual-asr 0.1.0 + fairseq2 0.6.
!pip install --quiet "omnilingual-asr==0.1.0" "fairseq2[arrow]==0.6" jiwer "datasets<4.0" soundfile
!pip uninstall -y torch torchaudio torchvision
!pip install --quiet torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0
print("\n>>> INSTALL TERMINE. Maintenant : Execution > Redemarrer la session, puis reprendre a la cellule 3.")

### ↓ APRÈS le redémarrage de session, reprends ici (cellules 3 → …)

In [ ]:
# Cellule 3 — fonctions d'eval + chargement du dataset (INDEPENDANT d'Omnilingual : re-executable)
import re, unicodedata, time, json
import numpy as np
import jiwer
from datasets import load_dataset

# --- normalisation TOLERANTE : minuscule + ponctuation ASCII retiree.
# On CONSERVE les caracteres baoule (ɛ ɔ ʼ, tons) : ils portent le sens. ---
_PUNCT = re.compile(r"[.,!?;:«»\"'`()\[\]{}…—–/\\]")
def normalize(s):
    s = unicodedata.normalize("NFC", (s or "")).lower().strip()
    s = _PUNCT.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

def score(refs, hyps):
    R, H = [], []
    for r, h in zip(refs, hyps):
        rn = normalize(r)
        if rn:                       # jiwer plante sur reference vide
            R.append(rn); H.append(normalize(h))
    return {"cer": float(jiwer.cer(R, H)), "wer": float(jiwer.wer(R, H)), "n": len(R)}

# Garde-fou "langue de sortie" (pour Simba) : proportion de mots-outils FRANCAIS.
_FR = {"le","la","les","de","des","du","un","une","et","est","que","qui","pour",
       "dans","avec","vous","nous","je","il","elle","ce","cette","sur","pas","ne"}
def looks_french(texts, k=40):
    words = " ".join(normalize(t) for t in texts[:k]).split()
    return (sum(w in _FR for w in words) / len(words)) if words else 0.0

def get_audio(row):
    a = row["audio"]
    if isinstance(a, dict):                         # datasets < 4.0
        return np.asarray(a["array"], dtype="float32"), int(a["sampling_rate"])
    samples = a.get_all_samples()                   # datasets >= 4.0 (AudioDecoder)
    return samples.data.numpy().squeeze().astype("float32"), int(samples.sample_rate)

# Dataset baoule (Common Voice bci) — miroir public CC0, revision epinglee, audio deja 16 kHz mono.
REV = "176d5f8ad6c04e5ca1a0e66770866709bcbb338b"
test = load_dataset("Klayt/baoule-common-voice", revision=REV, split="test")

LIMIT = None                    # mettre p.ex. 30 pour un essai rapide ; None = tout le split test (290)
rows = list(test)[:LIMIT] if LIMIT else list(test)
refs = [r["sentence"] for r in rows]
print(f"{len(rows)} clips baoule. Exemple: {refs[0]!r}")
w0, sr0 = get_audio(rows[0]); print("Audio:", w0.shape, sr0, "Hz")

# Chemin PORTABLE (relatif au dossier de travail) -> marche sur Colab (/content) ET Kaggle (/kaggle/working)
RESULTS_PATH = "asr_baoule_results.json"
def save(results):
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
def load_results():
    try:
        with open(RESULTS_PATH, encoding="utf-8") as f: return json.load(f)
    except FileNotFoundError:
        return {}

In [ ]:
# Cellule 4 — transcription OMNILINGUAL (M1 = 300M, M2 = 1B), ciblage bci_Latn
import os, tempfile, gc, torch, soundfile as sf
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs
assert "bci_Latn" in supported_langs, "bci_Latn absent de supported_langs !"

# ecrire les audios en WAV temporaires (le pipeline prend des chemins de fichiers)
tmp = tempfile.mkdtemp(); paths = []
for i, row in enumerate(rows):
    wav, sr = get_audio(row)
    p = os.path.join(tmp, f"{i}.wav"); sf.write(p, wav, sr); paths.append(p)

def run_omni(card, label):
    t0 = time.time(); pipe = ASRInferencePipeline(model_card=card); load_s = time.time()-t0
    t1 = time.time()
    hyps = pipe.transcribe(paths, lang=["bci_Latn"]*len(paths), batch_size=1)
    infer_s = time.time()-t1
    r = score(refs, hyps); r.update(load_s=round(load_s,1), infer_s=round(infer_s,1),
                                    samples=[{"ref": refs[i], "hyp": hyps[i]} for i in range(min(10,len(hyps)))])
    print(f"{label}: CER={r['cer']*100:.1f}%  WER={r['wer']*100:.1f}%  (charg {load_s:.0f}s, infer {infer_s:.0f}s)")
    del pipe; gc.collect(); torch.cuda.empty_cache()
    return r

results = load_results()
results["M1 Omnilingual 300M"] = run_omni("omniASR_CTC_300M", "M1 300M"); save(results)
results["M2 Omnilingual 1B"]   = run_omni("omniASR_CTC_1B",   "M2 1B");   save(results)
print("Sauvegarde ->", RESULTS_PATH)

## Étape 2 — Simba-S (M3, contrôle)

⚠️ Simba utilise `transformers` (stack différent d'Omnilingual). Si l'install ci-dessous
casse le runtime, **redémarre la session**, puis ré-exécute **la cellule 3** (dataset +
fonctions) et **reviens ici** — les résultats Omnilingual sont déjà sauvés sur disque.

**Rappel vérifié** : Simba n'a **aucun token de langue baoulé** ; il ne peut pas cibler le
baoulé et pourrait sortir du **français**. La cellule mesure ce risque (garde-fou).

In [ ]:
# Cellule 5 — install transformers pour Simba (n'installe PAS un autre torch)
!pip install --quiet "transformers>=4.40" sentencepiece protobuf
print(">>> Si erreur torch/fairseq2 plus bas : redemarre la session, re-exec cellule 3, puis reviens ici.")

In [ ]:
# Cellule 6 — transcription SIMBA-S + garde-fou "langue de sortie"
import torch
from transformers import pipeline as hf_pipeline

asr = hf_pipeline("automatic-speech-recognition", model="UBC-NLP/Simba-S",
                  device=0 if torch.cuda.is_available() else -1)

t0 = time.time(); hyps = []
for row in rows:
    wav, sr = get_audio(row)                       # deja 16 kHz mono
    hyps.append(asr({"array": wav, "sampling_rate": sr})["text"])
infer_s = time.time()-t0

fr = looks_french(hyps)
r = score(refs, hyps); r.update(infer_s=round(infer_s,1), fr_ratio=round(fr,2),
                                samples=[{"ref": refs[i], "hyp": hyps[i]} for i in range(min(10,len(hyps)))])
print(f"M3 Simba-S: CER={r['cer']*100:.1f}%  WER={r['wer']*100:.1f}%  (infer {infer_s:.0f}s)")
print(f"Ratio mots FRANCAIS dans la sortie: {fr*100:.0f}%")
if fr > 0.30:
    print("⚠️  Simba semble transcrire du FRANCAIS, pas du baoule -> a ECARTER pour le baoule.")
    r["verdict"] = "ecarte_langue_fr"
results = load_results(); results["M3 Simba-S"] = r; save(results)

In [ ]:
# Cellule 7 — tableau comparatif + exemples cote a cote
results = load_results()
if not results:
    print("Aucun resultat — lance d'abord les cellules de transcription (Omnilingual et/ou Simba).")
else:
    print(f"{'Modele':24s} {'CER':>7s} {'WER':>7s} {'n':>5s}  note")
    print("-"*60)
    for name, r in results.items():
        note = r.get("verdict","") or (f"fr={int(r['fr_ratio']*100)}%" if "fr_ratio" in r else "")
        print(f"{name:24s} {r['cer']*100:6.1f}% {r['wer']*100:6.1f}% {r['n']:5d}  {note}")

    print("\n=== Exemples (reference | hypotheses) ===")
    base = next(iter(results.values()))
    for i in range(min(8, len(base.get("samples", [])))):
        print(f"\n[{i}] REF   : {base['samples'][i]['ref']}")
        for name, r in results.items():
            if i < len(r.get("samples", [])):
                print(f"    {name:20s}: {r['samples'][i]['hyp']}")

    print("\nResultats complets ->", RESULTS_PATH,
          "\nReporter le tableau dans docs/benchmarks/0003-asr-baoule-evaluation-results.md")

## Conclusion

1. **Reporte** le tableau CER/WER (cellule 7) dans `docs/benchmarks/0003-asr-baoule-evaluation-results.md`.
2. Le **modèle au plus faible CER produisant réellement du baoulé** = base recommandée pour le fine-tuning (#474).
3. Cette mesure **informe** l'ADR « choix du modèle ASR baoulé » — elle ne le remplace pas.

*Seuils de lecture : cf. §5 du protocole 0003.*